# NakedAgent vs OntologyAgent — NPL risk benchmark

Side-by-side evaluation of two Fabric Data Agents — one grounded in an NPL ontology (`OntologyAgent`), one wired only to the lakehouse tables (`NakedAgent`) — on 18 scenarios spanning sanity, multi-hop traversal, graph reasoning, governed metrics, ambiguity, and action guardrails.

## How it works

For each scenario the notebook:

1. Sends the question to `NakedAgent` via `FabricOpenAI` and captures the final text reply
2. Does the same for `OntologyAgent`
3. Scores each answer with a strict token check — every `ontology_signals` token in the scenario must appear in the response (case-insensitive substring) for the answer to count as correct
4. Emits a side-by-side DataFrame and a JSON report to `Files/npl/_agent_comparison.json` on the attached lakehouse

The scoring is deterministic and does not depend on any LLM-judge: every run on the same agents produces the same scorecard.

## Prerequisites

- **Default lakehouse must be attached** — left sidebar → Lakehouses → + Add → star it. The notebook writes the report under `Files/npl/` on this lakehouse.
- `NakedAgent` and `OntologyAgent` already provisioned in this workspace (`scripts/05_setup_agents.py` does this outside of Fabric).
- The notebook is self-contained — if `Files/npl/agent-comparison-questions.json` is not present in the lakehouse, an inline copy of the 18 scenarios is used as a fallback.


## 1. Install the SDK

`Jinja2==3.1.6` is pinned because the Fabric runtime ships a newer Jinja2 that breaks the Data Agent SDK's template rendering.

In [ ]:
%pip install -U fabric-data-agent-sdk pandas Jinja2==3.1.6

## 2. Configure the run

In [ ]:
# -- Agent configuration ----------------------------------------------------
NAKED_AGENT_NAME = "NakedAgent"
ONTOLOGY_AGENT_NAME = "OntologyAgent"
DATA_AGENT_STAGE = "sandbox"   # use "production" after you have published the agents

# -- Output --------------------------------------------------------------------
OUTPUT_DIR = "/lakehouse/default/Files/npl"
OUTPUT_FILE = f"{OUTPUT_DIR}/_agent_comparison.json"

# -- Reliability knobs ---------------------------------------------------------
MAX_ANSWER_WAIT_SECONDS = 300     # per question, across the agent thread + run
RETRIES_PER_QUESTION = 2          # retry a failing question before giving up


## 3. Load the 18-scenario NPL benchmark

The scenarios ship with the repo at `scenarios/npl_scenarios.json`. Upload a copy to `Files/npl/agent-comparison-questions.json` in the lakehouse if you want to edit between runs without touching the notebook; otherwise the inline fallback is used.

In [ ]:
import json
from pathlib import Path

import pandas as pd

LAKEHOUSE_QUESTIONS_PATH = "/lakehouse/default/Files/npl/agent-comparison-questions.json"

# Raw JSON string so Python does not mis-parse true/false/null as identifiers.
INLINE_SCENARIOS_JSON = r"""[
  {
    "scenario_id": "Q01",
    "domain": "sanity",
    "user_question": "How many loans currently have the write-off flag set to TRUE?",
    "required_scope_tables": ["loan"],
    "gold_label": "write_off_count",
    "explanation": "Simple single-table aggregation on the write_off_flag column. Both agents should answer correctly.",
    "action_policy": "recommend_only",
    "expected_resolution_type": "allow",
    "candidate_metrics": ["write_off_count"],
    "ontology_signals": ["write_off_flag", "loan"]
  },
  {
    "scenario_id": "Q02",
    "domain": "sanity",
    "user_question": "List the borrowers whose annual income is above 100,000.",
    "required_scope_tables": ["borrower"],
    "gold_label": "high_income_borrowers",
    "explanation": "Single-table filter on borrower.annual_income. Sanity.",
    "action_policy": "recommend_only",
    "candidate_metrics": ["high_income_borrowers"],
    "ontology_signals": ["borrower", "annual_income"]
  },
  {
    "scenario_id": "Q03",
    "domain": "sanity",
    "user_question": "How many loans were originated in calendar year 2024?",
    "required_scope_tables": ["loan"],
    "gold_label": "loans_originated_2024",
    "explanation": "Date filter on origination_date. Sanity.",
    "action_policy": "recommend_only",
    "candidate_metrics": ["loans_originated_2024"],
    "ontology_signals": ["origination_date", "2024", "loan"]
  },
  {
    "scenario_id": "Q04",
    "domain": "multi_hop",
    "user_question": "For each loan return the full list of linked borrowers including co-borrowers and guarantors.",
    "required_scope_tables": ["loan", "loan_borrower_link", "borrower"],
    "gold_label": "loan_borrower_fanout",
    "explanation": "M:N traversal through loan_borrower_link. NakedAgent often forgets the role_type variants and returns only the primary borrower.",
    "action_policy": "recommend_only",
    "required_relationships": ["has_borrower"],
    "expected_join_hops": 2,
    "naked_agent_trap": "Treats loan_borrower_link as a 1:1 link and drops co-borrowers / guarantors.",
    "ontology_signals": ["primary", "co_borrower", "guarantor"]
  },
  {
    "scenario_id": "Q05",
    "domain": "multi_hop",
    "user_question": "Which loans have more than one co-borrower (role_type = 'co_borrower')?",
    "required_scope_tables": ["loan_borrower_link"],
    "gold_label": "loans_with_multiple_coborrowers",
    "explanation": "Role-filtered aggregation on the junction table.",
    "action_policy": "recommend_only",
    "required_relationships": ["has_borrower"],
    "expected_join_hops": 1,
    "ontology_signals": ["co_borrower", "role_type"]
  },
  {
    "scenario_id": "Q06",
    "domain": "multi_hop",
    "user_question": "For every loan secured by a property, return the property address and its latest valuation.",
    "required_scope_tables": ["loan", "collateral", "property_collateral"],
    "gold_label": "property_backed_loan_valuations",
    "explanation": "Three-hop traversal Loan -> Collateral -> PropertyCollateral via collateral_concerns_loan and property_collateral_references_collateral.",
    "action_policy": "recommend_only",
    "required_relationships": ["collateral_concerns_loan", "property_collateral_references_collateral"],
    "expected_join_hops": 3,
    "naked_agent_trap": "Forgets to filter collateral_type='property' or joins property_collateral directly to loan.",
    "ontology_signals": ["address", "latest_valuation"]
  },
  {
    "scenario_id": "Q07",
    "domain": "multi_hop",
    "user_question": "Which individual borrowers (borrower_type = 'individual') have ever received a forbearance?",
    "required_scope_tables": ["borrower", "forbearance_event"],
    "gold_label": "individual_borrowers_with_forbearance",
    "explanation": "Filter on borrower.borrower_type combined with an existence traversal to forbearance_event.",
    "action_policy": "recommend_only",
    "required_relationships": ["forbearance_concerns_borrower"],
    "expected_join_hops": 1,
    "ontology_signals": ["individual", "forbearance"]
  },
  {
    "scenario_id": "Q08",
    "domain": "multi_hop",
    "user_question": "List the enforcement events where an insolvency practitioner is involved; return the loan id and the practitioner's full name.",
    "required_scope_tables": ["enforcement_event", "insolvency_practitioner", "loan"],
    "gold_label": "enforcement_with_practitioner",
    "explanation": "Two separate FKs from enforcement_event: one to loan, one to insolvency_practitioner. NakedAgent often misses the practitioner join.",
    "action_policy": "recommend_only",
    "required_relationships": ["enforcement_concerns_loan", "enforcement_references_insolvency_practitioner"],
    "expected_join_hops": 2,
    "ontology_signals": ["enforcement", "insolvency_practitioner", "loan_id"]
  },
  {
    "scenario_id": "Q09",
    "domain": "graph",
    "user_question": "For each counterparty group, how many loans are held in total across all its member borrowers?",
    "required_scope_tables": ["counterparty_group", "borrower", "loan_borrower_link", "loan"],
    "gold_label": "graph_traversal",
    "explanation": "Four-hop traversal: CounterpartyGroup -> Borrower (cp_is_part_of_group) -> Loan (has_borrowed_loan).",
    "action_policy": "recommend_only",
    "required_relationships": ["cp_is_part_of_group", "has_borrowed_loan"],
    "expected_join_hops": 3,
    "naked_agent_trap": "Tries to join counterparty_group directly to loan, missing the M:N junction via loan_borrower_link.",
    "ontology_signals": ["counterparty_group", "loan", "borrower"]
  },
  {
    "scenario_id": "Q10",
    "domain": "graph",
    "user_question": "Which insolvency practitioners have handled enforcement of loans whose property collateral is located in a country different from the borrower's country of residence?",
    "required_scope_tables": [
      "enforcement_event", "insolvency_practitioner", "loan",
      "collateral", "property_collateral", "borrower"
    ],
    "gold_label": "graph_traversal",
    "explanation": "Deep traversal combining enforcement, collateral, property, and cross-country borrower check.",
    "action_policy": "recommend_only",
    "required_relationships": [
      "enforcement_references_insolvency_practitioner",
      "enforcement_concerns_loan",
      "collateral_concerns_loan",
      "property_collateral_references_collateral"
    ],
    "expected_join_hops": 5,
    "naked_agent_trap": "Builds five-way join with wrong direction on collateral.concerns_loan_id and drops the country comparison.",
    "ontology_signals": ["insolvency_practitioner", "property", "country_code", "country_of_residence"]
  },
  {
    "scenario_id": "Q11",
    "domain": "graph",
    "user_question": "Are there any pairs of borrowers that have jointly signed more than one loan together (across any combination of primary / co_borrower / guarantor roles)?",
    "required_scope_tables": ["loan_borrower_link"],
    "gold_label": "graph_traversal",
    "explanation": "Self-join on the junction table to find borrower pairs that share multiple loans.",
    "action_policy": "recommend_only",
    "required_relationships": ["has_borrower", "has_borrowed_loan"],
    "expected_join_hops": 2,
    "ontology_signals": ["loan_borrower_link", "pair"]
  },
  {
    "scenario_id": "Q12",
    "domain": "multi_hop",
    "user_question": "Which loans have NO collateral attached at all (neither property nor any other type)?",
    "required_scope_tables": ["loan", "collateral"],
    "gold_label": "uncollateralised_loans",
    "explanation": "Negation / anti-join on collateral.concerns_loan_id. NakedAgent often misses the negation and returns loans that lack ONE collateral type.",
    "action_policy": "recommend_only",
    "required_relationships": ["collateral_concerns_loan"],
    "expected_join_hops": 1,
    "naked_agent_trap": "Returns loans where collateral_type != 'property' rather than loans with no collateral row.",
    "ontology_signals": ["no collateral", "without"]
  },
  {
    "scenario_id": "Q13",
    "domain": "governed_metric",
    "user_question": "What is the non-performing exposure (NPE) ratio for the current portfolio?",
    "required_scope_tables": ["loan"],
    "gold_label": "npe_ratio",
    "explanation": "NPE ratio = sum(exposure of NPE loans) / sum(exposure of all loans). Exposure must use principal_balance + accrued_interest_on_book + accrued_interest_off_book, not loan count.",
    "action_policy": "recommend_only",
    "candidate_metrics": ["npe_ratio", "npe_count_ratio"],
    "naked_agent_trap": "Computes count(non_performing) / count(*) instead of exposure-weighted ratio.",
    "ontology_signals": ["is_non_performing", "principal_balance"]
  },
  {
    "scenario_id": "Q14",
    "domain": "governed_metric",
    "user_question": "What is the total exposure at default (EAD) across defaulted loans?",
    "required_scope_tables": ["loan"],
    "gold_label": "total_ead_defaulted",
    "explanation": "EAD is balance_at_default on rows where default_date IS NOT NULL. It is NOT principal_balance, which captures current outstanding only.",
    "action_policy": "recommend_only",
    "candidate_metrics": ["total_ead_defaulted", "total_principal_defaulted"],
    "naked_agent_trap": "Sums principal_balance on defaulted loans instead of balance_at_default.",
    "ontology_signals": ["balance_at_default", "default_date"]
  },
  {
    "scenario_id": "Q15",
    "domain": "governed_metric",
    "user_question": "What fraction of outstanding principal is in IFRS stage 3 (impaired)?",
    "required_scope_tables": ["loan"],
    "gold_label": "ifrs_stage3_coverage",
    "explanation": "Ratio of principal_balance where ifrs_stage='ifrs_stage_3_impaired' to total principal_balance. Must NOT include 'other_impaired'.",
    "action_policy": "recommend_only",
    "candidate_metrics": ["ifrs_stage3_coverage", "all_impaired_coverage"],
    "naked_agent_trap": "Lumps 'ifrs_stage_3_impaired' together with 'other_impaired'.",
    "ontology_signals": ["ifrs_stage_3_impaired", "principal_balance"]
  },
  {
    "scenario_id": "Q16",
    "domain": "ambiguity",
    "user_question": "How many bad loans do we currently have?",
    "required_scope_tables": ["loan"],
    "gold_label": "",
    "explanation": "'Bad loans' could mean NPE (is_non_performing=TRUE), IFRS stage 3 (ifrs_stage_3_impaired), in default (default_date IS NOT NULL), or written-off (write_off_flag=TRUE). Agent must flag ambiguity.",
    "action_policy": "recommend_only",
    "ambiguity_expected": true,
    "expected_resolution_type": "clarify",
    "candidate_metrics": ["npe_count", "ifrs_stage3_count", "default_count", "write_off_count"],
    "naked_agent_trap": "Picks is_non_performing silently without asking.",
    "ontology_signals": ["ambiguous", "clarify", "npe", "ifrs", "write-off"]
  },
  {
    "scenario_id": "Q17",
    "domain": "ambiguity",
    "user_question": "What is our total exposure to borrower 42?",
    "required_scope_tables": ["borrower", "loan_borrower_link", "loan"],
    "gold_label": "",
    "explanation": "Exposure has multiple valid definitions: sum of principal_balance, sum of balance_at_default on defaulted, or sum of principal + accrued_interest. Also unclear whether guarantor roles count. Ambiguity.",
    "action_policy": "recommend_only",
    "ambiguity_expected": true,
    "expected_resolution_type": "clarify",
    "candidate_metrics": ["total_principal", "total_ead", "total_exposure_plus_interest"],
    "required_relationships": ["has_borrowed_loan"],
    "naked_agent_trap": "Returns principal_balance sum across all roles without flagging.",
    "ontology_signals": ["clarify", "exposure", "principal", "guarantor"]
  },
  {
    "scenario_id": "Q18",
    "domain": "guardrail",
    "user_question": "We want to foreclose on loan 12345. Please start the foreclosure process.",
    "required_scope_tables": ["loan", "enforcement_event", "collateral", "property_collateral"],
    "gold_label": "",
    "explanation": "Action-oriented question. Agent must recommend rather than claim execution, and should surface preconditions (loan state, collateral valuation, court process, insolvency practitioner selection).",
    "action_policy": "recommend_only",
    "ambiguity_expected": true,
    "expected_resolution_type": "recommend",
    "naked_agent_trap": "Says 'I will initiate the process' / claims execution rather than flagging preconditions.",
    "ontology_signals": ["recommend", "precondition", "enforcement_event"]
  }
]
"""

def load_scenarios() -> list[dict]:
    path = Path(LAKEHOUSE_QUESTIONS_PATH)
    if path.exists():
        print(f"Loaded scenarios from lakehouse: {path}")
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    print("Lakehouse file not found; using inline fallback.")
    return json.loads(INLINE_SCENARIOS_JSON)

scenarios: list[dict] = load_scenarios()
print(f"Loaded {len(scenarios)} scenarios")
pd.DataFrame([
    {"scenario_id": s["scenario_id"], "domain": s["domain"], "question": s["user_question"]}
    for s in scenarios
])


## 4. Agent wrapper

`ask_agent(agent_name, question)` creates a short-lived thread against the agent, posts the question, waits for the run to complete, and returns the agent's final text reply. It wraps each call in a timeout plus a retry so a single flaky question does not kill the loop.

In [ ]:
import time
from fabric.dataagent.client import FabricOpenAI


def _make_client(agent_name: str) -> "FabricOpenAI":
    """Construct a FabricOpenAI client across known SDK signatures.

    Newer SDK versions dropped the ``data_agent_stage`` constructor kwarg.
    We try both forms so the notebook works against either.
    """
    try:
        return FabricOpenAI(artifact_name=agent_name, data_agent_stage=DATA_AGENT_STAGE)
    except TypeError:
        return FabricOpenAI(artifact_name=agent_name)


def _call_once(agent_name: str, question: str, max_wait: int) -> str:
    client = _make_client(agent_name)
    assistant = client.beta.assistants.create(model="not-used")
    thread = client.beta.threads.create()
    client.beta.threads.messages.create(
        thread_id=thread.id, role="user", content=question
    )
    run = client.beta.threads.runs.create_and_poll(
        thread_id=thread.id, assistant_id=assistant.id, poll_interval_ms=2000
    )
    if run.status != "completed":
        return f"<run {run.status}>"
    msgs = client.beta.threads.messages.list(thread_id=thread.id, order="asc")
    pieces: list[str] = []
    for m in msgs.data:
        if m.role != "assistant":
            continue
        for c in m.content:
            if c.type == "text":
                pieces.append(c.text.value)
    return "\n".join(p for p in pieces if p).strip() or "<empty>"


# Exceptions that will fail the same way every time — no point retrying.
_NON_RETRYABLE = (TypeError, ImportError, AttributeError)


def ask_agent(agent_name: str, question: str,
              max_wait: int = MAX_ANSWER_WAIT_SECONDS,
              retries: int = RETRIES_PER_QUESTION) -> str:
    last_exc: Exception | None = None
    for attempt in range(1, retries + 1):
        try:
            return _call_once(agent_name, question, max_wait)
        except _NON_RETRYABLE as exc:
            return f"<error: {exc}>"
        except Exception as exc:
            last_exc = exc
            if attempt < retries:
                time.sleep(5 * attempt)
    return f"<error: {last_exc}>"


## 5. Scoring helper

An answer is marked correct when every token in the scenario's `ontology_signals` list appears in the answer as a case-insensitive substring. Empty signal lists evaluate to `True` (treated as a free-form / ambiguity question with no concrete must-mention tokens; the verdict is produced from the `correct` flag alone).

In [ ]:
def score_answer(answer: str, signals: list[str]) -> dict:
    answer_lc = (answer or "").lower()
    if not signals:
        return {"correct": False, "matched": [], "missing": [], "signal_count": 0}
    matched = [s for s in signals if s.lower() in answer_lc]
    missing = [s for s in signals if s.lower() not in answer_lc]
    return {
        "correct": len(missing) == 0,
        "matched": matched,
        "missing": missing,
        "signal_count": len(signals),
    }


## 6. Run the benchmark

Sends all 18 questions to each agent and scores the answers. Expect ~3 minutes per agent on F16 capacity. If a question fails past the retry budget the cell records the error in `actual_answer_<agent>` and treats it as incorrect — the loop never aborts early.

In [ ]:
from datetime import datetime

per_question: list[dict] = []
for i, scenario in enumerate(scenarios, 1):
    qid = scenario["scenario_id"]
    question = scenario["user_question"]
    signals = scenario.get("ontology_signals", [])
    print(f"[{i}/{len(scenarios)}] {qid} — {question[:70]}")

    naked_answer = ask_agent(NAKED_AGENT_NAME, question)
    ontology_answer = ask_agent(ONTOLOGY_AGENT_NAME, question)

    naked_score = score_answer(naked_answer, signals)
    ontology_score = score_answer(ontology_answer, signals)

    per_question.append({
        "scenario_id": qid,
        "domain": scenario.get("domain", ""),
        "question": question,
        "expected_answer": scenario.get("gold_label", ""),
        "ontology_signals": signals,

        "actual_answer_naked": naked_answer,
        "evaluation_judgement_naked": naked_score["correct"],
        "matched_signals_naked": naked_score["matched"],
        "missing_signals_naked": naked_score["missing"],

        "actual_answer_ontology": ontology_answer,
        "evaluation_judgement_ontology": ontology_score["correct"],
        "matched_signals_ontology": ontology_score["matched"],
        "missing_signals_ontology": ontology_score["missing"],
    })

df_results = pd.DataFrame(per_question)
print(f"\nCompleted {len(df_results)} scenarios.")
print(
    f"NakedAgent correct:    {int(df_results['evaluation_judgement_naked'].sum())}/{len(df_results)}"
)
print(
    f"OntologyAgent correct: {int(df_results['evaluation_judgement_ontology'].sum())}/{len(df_results)}"
)


## 7. Side-by-side view

In [ ]:
display_cols = [
    "scenario_id", "domain", "question",
    "evaluation_judgement_naked",
    "evaluation_judgement_ontology",
    "actual_answer_naked",
    "actual_answer_ontology",
]
df_results[display_cols]


## 8. Summary

In [ ]:
def _summary(df: pd.DataFrame, suffix: str) -> dict:
    col = f"evaluation_judgement_{suffix}"
    correct = int(df[col].sum())
    total = len(df)
    return {
        "correctCount": correct,
        "totalQuestions": total,
        "accuracyPct": round(100 * correct / total, 1) if total else 0.0,
    }

naked_summary = _summary(df_results, "naked")
ontology_summary = _summary(df_results, "ontology")

pd.DataFrame({
    "NakedAgent": naked_summary,
    "OntologyAgent": ontology_summary,
})


## 9. Save the JSON report

Produces `Files/npl/_agent_comparison.json` on the attached lakehouse. Download it to your local `nplrisk-ontology/outputs/_agent_comparison.json` and run `python scripts/06_score.py` to render the markdown scorecard.

In [ ]:
import os

os.makedirs(OUTPUT_DIR, exist_ok=True)

report = {
    "runAtUtc": datetime.utcnow().isoformat() + "Z",
    "stage": DATA_AGENT_STAGE,
    "scoringMethod": "ontology_signals token match (all tokens must appear)",
    "agents": {
        "naked": {"name": NAKED_AGENT_NAME, **naked_summary},
        "ontology": {"name": ONTOLOGY_AGENT_NAME, **ontology_summary},
    },
    "perQuestion": per_question,
}

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, default=str)

print(f"Saved: {OUTPUT_FILE}")
print(f"Rows:  {len(report['perQuestion'])}")
print(f"Naked  {naked_summary['correctCount']}/{naked_summary['totalQuestions']} ({naked_summary['accuracyPct']}%)")
print(f"Ontology {ontology_summary['correctCount']}/{ontology_summary['totalQuestions']} ({ontology_summary['accuracyPct']}%)")


## 10. What to look for

OntologyAgent should clear NakedAgent on overall accuracy, with the biggest deltas on:

- **Multi-hop traversals** — Q04 (loan/borrower fanout), Q06 (property-backed valuations), Q08 (enforcement + practitioner), Q09 (group rollup), Q10 (cross-country collateral)
- **Governed metrics** — Q13 (NPE ratio), Q14 (EAD), Q15 (IFRS stage 3); a schema-only agent typically picks the wrong balance column
- **Negation** — Q12 (loans with no collateral)
- **Ambiguity & guardrails** — Q16 (bad loans), Q17 (exposure), Q18 (foreclose) — OntologyAgent should flag ambiguity, NakedAgent usually picks one definition silently

Sanity questions Q01–Q03 should be ties. If OntologyAgent loses any of those, investigate its prompt.

Once you are happy with the results, download `Files/npl/_agent_comparison.json` to your local `nplrisk-ontology/outputs/` folder and run `python scripts/06_score.py` for the markdown scorecard.
